# Local clinical extraction demo

Prepared synthetic walkthrough of the same steps as `run.py`.  
Not a clinical validation claim. Default path makes **no model call**.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import demo_viz as viz

viz.show(viz.mode_banner(prepared=True))
viz.show(viz.cli_cheatsheet())

## Act I — Deep letter (seizure frequency)

Teaching letter `TEACH-GAN-01`: a typical monthly pattern competes with a year-to-date count.

In [ ]:
deep = viz.load_json("deep_letter.json")
viz.show(viz.letter_html(deep["text"], title=deep["id"]))
print(deep["gold_note"])

In [ ]:
# Prepared result — same public shape as CLI seizure-frequency output.
result = deep["result"]
viz.show(viz.result_card(result, title="Prepared seizure-frequency result"))
viz.show(viz.letter_html(deep["text"], evidence=result.get("evidence"), title="Evidence in the letter"))

In [ ]:
viz.show(viz.stage_stepper(deep["stages"]))
viz.show(viz.punchline_compare(deep["model_first"], deep["result"]))

## Glance batch (2–3 notes)

Same workflow, smaller letters — what a multi-row `results.jsonl` feels like.

In [ ]:
glance_notes = viz.load_jsonl("glance_notes.jsonl")
glance = viz.load_json("glance_results.json")
assert len(glance_notes) == len(glance["rows"])
viz.show(viz.batch_table(glance["rows"]))
for note, row in zip(glance_notes, glance["rows"]):
    print(f"{note['id']}: {note['text'][:72]}… → {row['result']['value']}")

In [ ]:
viz.show(viz.files_footer(output_name="results.jsonl"))

## Act II — Live endpoint (optional)

Uses the same `.env` as the CLI. Skip this section if you only want the prepared demo.

Set `RUN_LIVE = True` after `Copy-Item .env.example .env` (or `cp`) and editing `VLLM_*`.

In [ ]:
RUN_LIVE = False  # flip to True to call your endpoint
extractor = None

def _load_dotenv(path: Path) -> None:
    import os

    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; copy .env.example to .env first.")
    for number, raw in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key.strip():
            raise ValueError(f".env line {number}: expected KEY=value")
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        os.environ.setdefault(key.strip(), value)

if not RUN_LIVE:
    viz.show(viz.mode_banner(prepared=True))
    print("Live cells skipped. Set RUN_LIVE = True to probe your endpoint.")
else:
    _load_dotenv(ROOT / ".env")
    from clinical_extraction_local import ClinicalExtractor, VLLMClient
    from clinical_extraction_local import _disable_dspy_cache, _prepare_internal_import
    from clinical_extraction_local.config import EndpointConfig

    _prepare_internal_import()
    _disable_dspy_cache()
    config = EndpointConfig.from_env()
    viz.show(viz.mode_banner(prepared=False))
    print(config.public_dict())
    client = VLLMClient(config)
    extractor = ClinicalExtractor(client, config.settings)
    check = extractor.run_workflow(
        "seizure_frequency",
        note_id="synthetic-endpoint-check",
        text="Synthetic note: the patient currently has two seizures per month.",
    )
    print({
        "requested_model": check.model_response.requested_model,
        "response_model": check.model_response.response_model,
        "json_mode": check.model_response.structured_output_mode,
        "value": check.result.get("value"),
    })
    live = extractor.seizure_frequency(note_id=deep["id"], text=deep["text"])
    viz.show(viz.result_card(live, title="Live seizure-frequency result"))
    viz.show(viz.letter_html(deep["text"], evidence=live.get("evidence"), title="Live evidence"))
    print("Prepared answer:", deep["result"]["value"], "| Live answer:", live.get("value"))


## Act III — Clinical findings (second workflow)

Separate contract: Diagnosis, Seizure Frequency, Prescription, Investigations.  
These seizure-frequency findings are **not** the Gan-derived single current answer.

In [ ]:
findings = viz.load_json("findings_letter.json")
viz.show(viz.letter_html(findings["text"], title=findings["id"]))
print(findings["note"])
viz.show(viz.findings_cards(findings["result"]))

In [ ]:
# Optional live findings — only runs when Act II set RUN_LIVE = True and succeeded.
if RUN_LIVE and extractor is not None:
    live_findings = extractor.clinical_findings(note_id=findings["id"], text=findings["text"])
    viz.show(viz.findings_cards(live_findings))
else:
    print("Prepared findings only. Re-run Act II with RUN_LIVE = True for a live four-family call.")


## Next

1. CLI: `python run.py seizure-frequency --input examples/seizure_frequency/notes.jsonl --output results.jsonl`
2. Read `docs/OUTPUTS.md` and `docs/PRIVATE_DATA.md` before real notes.
3. Troubleshooting: `docs/TROUBLESHOOTING.md`.